# DeepTrace — generate diffusion faces (cross-generator TEST set)

**Run on:** Kaggle Notebook (free) · Accelerator **GPU T4 x2 or P100** · Internet **On**

Models (both free & open):
- **SDXL base 1.0** — CreativeML Open RAIL++-M license.
- **FLUX.1 [schnell]** — Apache-2.0. Much larger download (~34 GB); loaded 4-bit to fit 16 GB.

These images are **never used for training** — they measure whether a detector trained on StyleGAN
generalises to modern generators.

**How to run (recommended):** set `MODELS` below, then **Save Version → Save & Run All (Commit)**.
It runs in the background (Kaggle's per-run limit is 12 h) and everything in `/kaggle/working` is kept
as the version's output. Each model runs in its own process, so GPU memory is freed between them.

**Splitting across runs** (free GPU quota is weekly): generation is resumable. Attach the previous
version's output as an input, set `PREVIOUS_OUTPUT` to it, and commit again — existing images are
copied in and generation continues from the next index.

If a Hugging Face model page asks you to accept terms, accept it on the website, then add your HF
token as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets).

In [ ]:
# ---- Setup: clone your repo + install the few extras Kaggle doesn't ship ----
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Vuday3336/DeepTrace-AI-face-detector.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "deeptrace"
ML = REPO / "ml"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(ML)
print("Working in", Path.cwd())

# ---- Parameters (set BEFORE the installs: FLUX needs an extra package that SDXL must not have) ----
MODELS = ["sdxl"]            # ["sdxl"], ["flux-schnell"], or both (both may not fit in one 12 h run)
COUNT_PER_MODEL = 500        # a bit above 400 leaves room for images where MTCNN finds no face
PREVIOUS_OUTPUT = None       # e.g. Path("/kaggle/input/<previous-version-output>/generated")

# bitsandbytes is installed ONLY for FLUX. With SDXL it is not merely useless but harmful: older
# builds import `triton.ops`, removed in Triton 3.x, which makes `import diffusers` fail outright.
requirements = "requirements/generation-flux.txt" if "flux-schnell" in MODELS else "requirements/generation.txt"
print("Installing", requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

try:  # optional Hugging Face login from a Kaggle secret
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception as exc:  # no secret configured -> public models still work
    print("No HF login:", type(exc).__name__)

OUT = WORK / "generated"

import shutil
if PREVIOUS_OUTPUT is not None:
    if not Path(PREVIOUS_OUTPUT).is_dir():
        raise FileNotFoundError(
            f"PREVIOUS_OUTPUT does not exist: {PREVIOUS_OUTPUT}. "
            "Attach the dataset first (right panel -> Add Input) and check the exact path, "
            "or set PREVIOUS_OUTPUT = None for a first run."
        )
    shutil.copytree(PREVIOUS_OUTPUT, OUT, dirs_exist_ok=True)
    print("Resumed from", PREVIOUS_OUTPUT, "->", len(list(OUT.rglob("*.png"))), "images carried over")

## Generate

In [ ]:
for model in MODELS:
    subprocess.run([sys.executable, "scripts/generate_diffusion_faces.py", "--model", model,
                    "--count", str(COUNT_PER_MODEL), "--out", str(OUT)], check=True)
    # If FLUX outputs are black/degenerate, add "--flux-dtype", "bfloat16" above and rerun.

In [ ]:
# Contact sheet of the first images — check they look like photos of single faces
from PIL import Image
from IPython.display import display

def contact_sheet(folder, n=12, size=192):
    files = sorted(Path(folder).glob("*.png"))[:n]
    sheet = Image.new("RGB", (size * 6, size * ((len(files) + 5) // 6)))
    for i, f in enumerate(files):
        sheet.paste(Image.open(f).convert("RGB").resize((size, size)), ((i % 6) * size, (i // 6) * size))
    return sheet

for model in MODELS:
    if (OUT / model).exists():
        print(model); display(contact_sheet(OUT / model))

## Save as a dataset

When the committed run finishes, open the version → **Output** → **New Dataset**, named
`deeptrace-generated` (it should contain `sdxl/` and/or `flux-schnell/`). Attach it in
`01_data_prep_and_audit` (section 1b). The `metadata.jsonl` files (prompt, seed, settings per image)
make every image reproducible, and also record how many outputs were degenerate.